In [16]:
import json
import os
import sqlite3
import subprocess
from pymongo import MongoClient
import pandas as pd

In [40]:
def create_ozon_table(conn):
    """
    Создаёт таблицу ozon с заданной структурой.
    Параметры:
        conn: соединение sqlite3
    """
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS ozon (
            uid INTEGER PRIMARY KEY AUTOINCREMENT,
            image_href TEXT,
            name TEXT,
            cost INTEGER,
            discount INTEGER,
            rating REAL,
            reviews INTEGER,
            category_name TEXT,
            page_url TEXT,
            page_number INTEGER
        )
    ''')
    conn.commit()
    print("Таблица ozon создана (или уже существовала).")

def drop_ozon_table(conn):
    """
    Удаляет таблицу ozon, если она существует.
    """
    cursor = conn.cursor()
    cursor.execute('DROP TABLE IF EXISTS ozon')
    conn.commit()
    print("Таблица ozon удалена.")

def fill_ozon_table_from_json(conn, json_path):
    """
    Заполняет таблицу ozon данными из JSON-файла.
    Ожидается, что JSON содержит список объектов с полями:
    image_href, name, cost, discount, rating, reviews, category_name, page_url, page_number
    """
    if not os.path.exists(json_path):
        print(f"Файл {json_path} не найден.")
        return

    with open(json_path, 'r', encoding='utf-8') as f:
        products = json.load(f)

    if not isinstance(products, list):
        print("JSON должен содержать список объектов.")
        return

    cursor = conn.cursor()

    # Подготовка запроса
    insert_sql = '''
        INSERT INTO ozon (image_href, name, cost, discount, rating, reviews, category_name, page_url, page_number)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    '''

    data = []
    for item in products:
        # Извлекаем нужные поля, подставляя None, если ключа нет
        row = (
            item.get('image_href'),
            item.get('name'),
            item.get('cost'),
            item.get('discount'),
            item.get('rating'),
            item.get('reviews'),
            item.get('category_name'),
            item.get('page_url'),
            item.get('page_number')
        )
        data.append(row)

    cursor.executemany(insert_sql, data)
    conn.commit()
    print(f"Вставлено записей: {len(data)}")

def get_high_discount_high_rating(conn):
    """
    Выводит записи из таблицы ozon, у которых discount > 50 и rating > 4.5.
    """
    cursor = conn.cursor()
    
    cursor.execute("SELECT * FROM ozon WHERE discount > 50 AND rating > 4.5")
    rows = cursor.fetchall()
    
    if not rows:
        print("Записей, удовлетворяющих условию, не найдено.")
    else:
        # Получаем имена столбцов
        col_names = [description[0] for description in cursor.description]
        return rows, col_names
        # print(f"Найде но записей: {len(rows)}\n")
        # for i, row in enumerate(rows, 1):
        #     print(f"--- Запись {i} ---")
        #     for name, value in zip(col_names, row):
        #         print(f"{name}: {value}")
        #     print()

In [41]:
db_path = "train.db"
conn = sqlite3.connect(db_path)
drop_ozon_table(conn)
conn.close()

Таблица ozon удалена.


In [42]:
db_path = "train.db"
conn = sqlite3.connect(db_path)

create_ozon_table(conn)
fill_ozon_table_from_json(conn, "ozon_products.json")
data, cols = get_high_discount_high_rating(conn)
df = pd.DataFrame(data, columns=cols)
# drop_ozon_table(conn)
conn.close()
print(df.info())
df

Таблица ozon создана (или уже существовала).
Вставлено записей: 1485
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1023 entries, 0 to 1022
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   uid            1023 non-null   int64  
 1   image_href     1023 non-null   object 
 2   name           1023 non-null   object 
 3   cost           1023 non-null   int64  
 4   discount       1023 non-null   int64  
 5   rating         1023 non-null   float64
 6   reviews        1023 non-null   int64  
 7   category_name  1023 non-null   object 
 8   page_url       1023 non-null   object 
 9   page_number    1023 non-null   int64  
dtypes: float64(1), int64(5), object(4)
memory usage: 80.1+ KB
None


,uid,image_href,name,cost,discount,rating,reviews,category_name,page_url,page_number
0,1,/product/novinka-konsol-igrovaya-pristavka-dly...,Новинка! Консоль игровая приставка для телевиз...,2029,79,4.6,23,Электроника,https://www.ozon.ru/category/elektronika-15500...,1
1,2,/product/kamera-videonablyudeniya-wifi-ulichna...,"Камера видеонаблюдения wifi, уличная, видеокам...",2890,81,4.8,8,Электроника,https://www.ozon.ru/category/elektronika-15500...,1
2,4,/product/umnaya-svetodiodnaya-lampochka-rgb-e2...,"Умная светодиодная лампочка RGB Е27 с Wi-Fi, Я...",656,73,4.8,30,Электроника,https://www.ozon.ru/category/elektronika-15500...,1
3,5,/product/oppo-smartfon-oppo-a6-pro-rostest-eac...,OPPO Смартфон OPPO A6 Pro Ростест (EAC) 8/256 ...,17214,56,4.9,12,Электроника,https://www.ozon.ru/category/elektronika-15500...,1
4,6,/product/propellery-dlya-fpv-drona-7-dyuymov-7...,Пропеллеры для FPV дрона 7 дюймов 7040 3х лопа...,383,57,4.9,7,Электроника,https://www.ozon.ru/category/elektronika-15500...,1
...,...,...,...,...,...,...,...,...,...,...
1018,1479,/product/milen-farmer-mp3-disk-3034740452/?at=...,Милен Фармер - мп3 диск,265,66,4.8,35,Музыка и видео,https://www.ozon.ru/category/muzyka-i-video-13...,2
1019,1480,/product/fleshka-4-gb-mr3-zarubezhnye-80-90-h-...,"Флешка 4 гб. ""Мр3 Зарубежные 80-90-х - 2018 """,1040,74,4.9,122,Музыка и видео,https://www.ozon.ru/category/muzyka-i-video-13...,2
1020,1481,/product/depeche-mode-greatest-hits-2017-2023-...,Depeche Mode - Greatest Hits (2017/2023) Переи...,565,87,4.8,22,Музыка и видео,https://www.ozon.ru/category/muzyka-i-video-13...,2
1021,1484,/product/depeche-mode-m-memento-mori-mexico-ci...,"DEPECHE MODE ""M"" (Memento Mori: Mexico City) B...",199,83,4.9,43,Музыка и видео,https://www.ozon.ru/category/muzyka-i-video-13...,2


In [39]:
def get_default_gateway_ip():
    """Возвращает IP шлюза по умолчанию (адрес хоста Windows в WSL)"""
    try:
        result = subprocess.run(
            ["ip", "route", "show", "default"],
            capture_output=True, text=True
        )
        for line in result.stdout.splitlines():
            if "default via" in line:
                parts = line.split()
                for i, part in enumerate(parts):
                    if part == "via" and i+1 < len(parts):
                        return parts[i+1]
    except Exception:
        pass
    return None

def get_mongo_client():
    """
    Возвращает клиент MongoDB, подключённый к серверу на Windows.
    Сначала пытается через шлюз по умолчанию,
    затем через host.docker.internal, затем localhost.
    """
    # 1. Пробуем IP шлюза по умолчанию
    gw_ip = get_default_gateway_ip()
    if gw_ip:
        try:
            client = MongoClient(f'mongodb://{gw_ip}:27017/', serverSelectionTimeoutMS=2000)
            client.admin.command('ping')
            print(f"Подключено через шлюз {gw_ip}")
            return client
        except Exception:
            pass

    # 2. Пробуем host.docker.internal 
    try:
        client = MongoClient('mongodb://host.docker.internal:27017/', serverSelectionTimeoutMS=2000)
        client.admin.command('ping')
        print("Подключено через host.docker.internal")
        return client
    except Exception:
        pass

    # 3. Пробуем localhost 
    try:
        client = MongoClient('mongodb://localhost:27017/', serverSelectionTimeoutMS=2000)
        client.admin.command('ping')
        print("Подключено через localhost")
        return client
    except Exception:
        pass

    raise ConnectionError("Не удалось подключиться к MongoDB. Проверьте, что сервер запущен и доступен.")

def create_ozon_collection(db, collection_name='ozon'):
    """
    Создаёт коллекцию в MongoDB, если она не существует.
    В MongoDB коллекции создаются автоматически при вставке первого документа,
    но можно создать явно для контроля.
    """
    # Проверяем, существует ли коллекция
    if collection_name not in db.list_collection_names():
        db.create_collection(collection_name)
        print(f"Коллекция '{collection_name}' создана.")
    else:
        print(f"Коллекция '{collection_name}' уже существует.")

def drop_ozon_collection(db, collection_name='ozon'):
    """
    Удаляет коллекцию из MongoDB.
    """
    db.drop_collection(collection_name)
    print(f"Коллекция '{collection_name}' удалена.")

def fill_ozon_collection_from_json(db, json_path, collection_name='ozon'):
    """
    Заполняет коллекцию данными из JSON-файла.
    Ожидается, что JSON содержит список объектов с полями:
    image_href, name, cost, discount, rating, reviews, category_name, page_url, page_number.
    Каждый документ получит автоматически сгенерированное поле _id.
    """
    if not os.path.exists(json_path):
        print(f"Файл {json_path} не найден.")
        return

    with open(json_path, 'r', encoding='utf-8') as f:
        products = json.load(f)

    if not isinstance(products, list):
        print("JSON должен содержать список объектов.")
        return

    collection = db[collection_name]
    # Вставляем все документы (каждый документ — это словарь)
    result = collection.insert_many(products)
    print(f"Вставлено документов: {len(result.inserted_ids)}")

def get_high_discount_high_rating_mongo(db, collection_name='ozon'):
    """
    Выполняет запрос к коллекции MongoDB и возвращает записи с discount > 50 и rating > 4.5.
    """
    collection = db[collection_name]
    query = {
        'discount': {'$gt': 50},
        'rating': {'$gt': 4.5}
    }
    cursor = collection.find(query)
    data = list(cursor)
    if data:
        columns = list(data[0].keys())
    else:
        columns = []
    return data, columns

In [49]:
client = get_mongo_client()
db = client['ozon_db']
drop_ozon_collection(db, 'ozon_products')
client.close()

Подключено через шлюз 172.25.208.1
Коллекция 'ozon_products' удалена.


In [50]:
client = get_mongo_client()

db = client['ozon_db']

# Создаём коллекцию
create_ozon_collection(db, 'ozon_products')

# Заполняем из JSON
fill_ozon_collection_from_json(db, 'ozon_products.json', 'ozon_products')
data, cols = get_high_discount_high_rating_mongo(db, 'ozon_products')
client.close()
df = pd.DataFrame(data, columns=cols)
print(df.info())
df


Подключено через шлюз 172.25.208.1
Коллекция 'ozon_products' создана.
Вставлено документов: 1485
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1023 entries, 0 to 1022
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   _id            1023 non-null   object 
 1   image_href     1023 non-null   object 
 2   name           1023 non-null   object 
 3   cost           1023 non-null   int64  
 4   discount       1023 non-null   int64  
 5   rating         1023 non-null   float64
 6   reviews        1023 non-null   int64  
 7   category_name  1023 non-null   object 
 8   page_url       1023 non-null   object 
 9   page_number    1023 non-null   int64  
dtypes: float64(1), int64(4), object(5)
memory usage: 80.1+ KB
None


,_id,image_href,name,cost,discount,rating,reviews,category_name,page_url,page_number
0,69bfc797413f9a03e7650689,/product/novinka-konsol-igrovaya-pristavka-dly...,Новинка! Консоль игровая приставка для телевиз...,2029,79,4.6,23,Электроника,https://www.ozon.ru/category/elektronika-15500...,1
1,69bfc797413f9a03e765068a,/product/kamera-videonablyudeniya-wifi-ulichna...,"Камера видеонаблюдения wifi, уличная, видеокам...",2890,81,4.8,8,Электроника,https://www.ozon.ru/category/elektronika-15500...,1
2,69bfc797413f9a03e765068c,/product/umnaya-svetodiodnaya-lampochka-rgb-e2...,"Умная светодиодная лампочка RGB Е27 с Wi-Fi, Я...",656,73,4.8,30,Электроника,https://www.ozon.ru/category/elektronika-15500...,1
3,69bfc797413f9a03e765068d,/product/oppo-smartfon-oppo-a6-pro-rostest-eac...,OPPO Смартфон OPPO A6 Pro Ростест (EAC) 8/256 ...,17214,56,4.9,12,Электроника,https://www.ozon.ru/category/elektronika-15500...,1
4,69bfc797413f9a03e765068e,/product/propellery-dlya-fpv-drona-7-dyuymov-7...,Пропеллеры для FPV дрона 7 дюймов 7040 3х лопа...,383,57,4.9,7,Электроника,https://www.ozon.ru/category/elektronika-15500...,1
...,...,...,...,...,...,...,...,...,...,...
1018,69bfc797413f9a03e7650c4f,/product/milen-farmer-mp3-disk-3034740452/?at=...,Милен Фармер - мп3 диск,265,66,4.8,35,Музыка и видео,https://www.ozon.ru/category/muzyka-i-video-13...,2
1019,69bfc797413f9a03e7650c50,/product/fleshka-4-gb-mr3-zarubezhnye-80-90-h-...,"Флешка 4 гб. ""Мр3 Зарубежные 80-90-х - 2018 """,1040,74,4.9,122,Музыка и видео,https://www.ozon.ru/category/muzyka-i-video-13...,2
1020,69bfc797413f9a03e7650c51,/product/depeche-mode-greatest-hits-2017-2023-...,Depeche Mode - Greatest Hits (2017/2023) Переи...,565,87,4.8,22,Музыка и видео,https://www.ozon.ru/category/muzyka-i-video-13...,2
1021,69bfc797413f9a03e7650c54,/product/depeche-mode-m-memento-mori-mexico-ci...,"DEPECHE MODE ""M"" (Memento Mori: Mexico City) B...",199,83,4.9,43,Музыка и видео,https://www.ozon.ru/category/muzyka-i-video-13...,2


In [37]:
def upsert_ozon_sqlite(conn, json_path):
    """
    Обновляет или вставляет записи в таблицу ozon из JSON-файла.
    Ключ для обновления: image_href (должен быть уникальным).
    """
    if not os.path.exists(json_path):
        print(f"Файл {json_path} не найден.")
        return

    with open(json_path, 'r', encoding='utf-8') as f:
        products = json.load(f)

    if not isinstance(products, list):
        print("JSON должен содержать список объектов.")
        return
    
    cursor = conn.cursor()

    insert_sql = '''
        INSERT OR REPLACE INTO ozon (image_href, name, cost, discount, rating, reviews, category_name, page_url, page_number)
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?)
    '''

    data = []
    for item in products:
        row = (
            item.get('image_href'),
            item.get('name'),
            item.get('cost'),
            item.get('discount'),
            item.get('rating'),
            item.get('reviews'),
            item.get('category_name'),
            item.get('page_url'),
            item.get('page_number')
        )
        data.append(row)

    cursor.executemany(insert_sql, data)
    conn.commit()
    print(f"Обновлено/вставлено записей: {len(data)}")


def upsert_ozon_mongo(db, collection_name, json_path):
    """
    Обновляет или вставляет документы в коллекцию MongoDB из JSON-файла.
    Ключ для обновления: image_href.
    """
    if not os.path.exists(json_path):
        print(f"Файл {json_path} не найден.")
        return

    with open(json_path, 'r', encoding='utf-8') as f:
        products = json.load(f)

    if not isinstance(products, list):
        print("JSON должен содержать список объектов.")
        return

    collection = db[collection_name]

    # Для ускорения можно создать индекс на поле image_href, если его нет
    collection.create_index('image_href', unique=True)

    inserted = 0
    updated = 0
    for item in products:
        # Используем image_href в качестве фильтра
        filter = {'image_href': item.get('image_href')}
        update = {'$set': item}  # обновляем все поля
        result = collection.update_one(filter, update, upsert=True)
        if result.upserted_id:
            inserted += 1
        elif result.modified_count:
            updated += 1

    print(f"Вставлено новых документов: {inserted}, обновлено: {updated}")

In [43]:
db_path = "train.db"
conn = sqlite3.connect(db_path)

upsert_ozon_sqlite(conn, "ozon_products_new.json")
data, cols = get_high_discount_high_rating(conn)
df = pd.DataFrame(data, columns=cols)

conn.close()
print(df.info())
df


Обновлено/вставлено записей: 15960
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10992 entries, 0 to 10991
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   uid            10992 non-null  int64  
 1   image_href     10992 non-null  object 
 2   name           10992 non-null  object 
 3   cost           10992 non-null  int64  
 4   discount       10992 non-null  int64  
 5   rating         10992 non-null  float64
 6   reviews        10992 non-null  int64  
 7   category_name  10992 non-null  object 
 8   page_url       10992 non-null  object 
 9   page_number    10992 non-null  int64  
dtypes: float64(1), int64(5), object(4)
memory usage: 858.9+ KB
None


,uid,image_href,name,cost,discount,rating,reviews,category_name,page_url,page_number
0,1,/product/novinka-konsol-igrovaya-pristavka-dly...,Новинка! Консоль игровая приставка для телевиз...,2029,79,4.6,23,Электроника,https://www.ozon.ru/category/elektronika-15500...,1
1,2,/product/kamera-videonablyudeniya-wifi-ulichna...,"Камера видеонаблюдения wifi, уличная, видеокам...",2890,81,4.8,8,Электроника,https://www.ozon.ru/category/elektronika-15500...,1
2,4,/product/umnaya-svetodiodnaya-lampochka-rgb-e2...,"Умная светодиодная лампочка RGB Е27 с Wi-Fi, Я...",656,73,4.8,30,Электроника,https://www.ozon.ru/category/elektronika-15500...,1
3,5,/product/oppo-smartfon-oppo-a6-pro-rostest-eac...,OPPO Смартфон OPPO A6 Pro Ростест (EAC) 8/256 ...,17214,56,4.9,12,Электроника,https://www.ozon.ru/category/elektronika-15500...,1
4,6,/product/propellery-dlya-fpv-drona-7-dyuymov-7...,Пропеллеры для FPV дрона 7 дюймов 7040 3х лопа...,383,57,4.9,7,Электроника,https://www.ozon.ru/category/elektronika-15500...,1
...,...,...,...,...,...,...,...,...,...,...
10987,17436,/product/karaoke-luchshie-via-sssr-100-pesen-d...,Караоке Лучшие ВИА СССР 100 песен DVD диск,357,60,4.8,320,Музыка и видео,https://www.ozon.ru/category/muzyka-i-video-13...,5
10988,17439,/product/laskovyy-may-sbornik-best-cd-v-karton...,Ласковый май - Сборник Best (CD в картонном ко...,185,53,4.8,1,Музыка и видео,https://www.ozon.ru/category/muzyka-i-video-13...,6
10989,17440,/product/odinokiy-pastuh-2-instrumentalnaya-mu...,Одинокий пастух 2 ИНСТРУМЕНТАЛЬНАЯ МУЗЫКА Мело...,248,58,5.0,5,Музыка и видео,https://www.ozon.ru/category/muzyka-i-video-13...,6
10990,17442,/product/depeche-mode-novye-i-luchshie-hity-cd...,"Depeche Mode Новые и лучшие хиты (CD-диск, сбо...",248,58,4.7,17,Музыка и видео,https://www.ozon.ru/category/muzyka-i-video-13...,6


In [51]:
client = get_mongo_client()
db = client['ozon_db']
upsert_ozon_mongo(db, 'ozon_products', "ozon_products_new.json")
data, cols = get_high_discount_high_rating_mongo(db, 'ozon_products')
df = pd.DataFrame(data, columns=cols)
client.close()
print(df.info())
df

Подключено через шлюз 172.25.208.1
Вставлено новых документов: 15960, обновлено: 0
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10992 entries, 0 to 10991
Data columns (total 10 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   _id            10992 non-null  object 
 1   image_href     10992 non-null  object 
 2   name           10992 non-null  object 
 3   cost           10992 non-null  int64  
 4   discount       10992 non-null  int64  
 5   rating         10992 non-null  float64
 6   reviews        10992 non-null  int64  
 7   category_name  10992 non-null  object 
 8   page_url       10992 non-null  object 
 9   page_number    10992 non-null  int64  
dtypes: float64(1), int64(4), object(5)
memory usage: 858.9+ KB
None


,_id,image_href,name,cost,discount,rating,reviews,category_name,page_url,page_number
0,69bfc797413f9a03e7650689,/product/novinka-konsol-igrovaya-pristavka-dly...,Новинка! Консоль игровая приставка для телевиз...,2029,79,4.6,23,Электроника,https://www.ozon.ru/category/elektronika-15500...,1
1,69bfc797413f9a03e765068a,/product/kamera-videonablyudeniya-wifi-ulichna...,"Камера видеонаблюдения wifi, уличная, видеокам...",2890,81,4.8,8,Электроника,https://www.ozon.ru/category/elektronika-15500...,1
2,69bfc797413f9a03e765068c,/product/umnaya-svetodiodnaya-lampochka-rgb-e2...,"Умная светодиодная лампочка RGB Е27 с Wi-Fi, Я...",656,73,4.8,30,Электроника,https://www.ozon.ru/category/elektronika-15500...,1
3,69bfc797413f9a03e765068d,/product/oppo-smartfon-oppo-a6-pro-rostest-eac...,OPPO Смартфон OPPO A6 Pro Ростест (EAC) 8/256 ...,17214,56,4.9,12,Электроника,https://www.ozon.ru/category/elektronika-15500...,1
4,69bfc797413f9a03e765068e,/product/propellery-dlya-fpv-drona-7-dyuymov-7...,Пропеллеры для FPV дрона 7 дюймов 7040 3х лопа...,383,57,4.9,7,Электроника,https://www.ozon.ru/category/elektronika-15500...,1
...,...,...,...,...,...,...,...,...,...,...
10987,69bfc7a57ad1dfd190c3a928,/product/karaoke-luchshie-via-sssr-100-pesen-d...,Караоке Лучшие ВИА СССР 100 песен DVD диск,357,60,4.8,320,Музыка и видео,https://www.ozon.ru/category/muzyka-i-video-13...,5
10988,69bfc7a57ad1dfd190c3a92b,/product/laskovyy-may-sbornik-best-cd-v-karton...,Ласковый май - Сборник Best (CD в картонном ко...,185,53,4.8,1,Музыка и видео,https://www.ozon.ru/category/muzyka-i-video-13...,6
10989,69bfc7a57ad1dfd190c3a92c,/product/odinokiy-pastuh-2-instrumentalnaya-mu...,Одинокий пастух 2 ИНСТРУМЕНТАЛЬНАЯ МУЗЫКА Мело...,248,58,5.0,5,Музыка и видео,https://www.ozon.ru/category/muzyka-i-video-13...,6
10990,69bfc7a57ad1dfd190c3a92e,/product/depeche-mode-novye-i-luchshie-hity-cd...,"Depeche Mode Новые и лучшие хиты (CD-диск, сбо...",248,58,4.7,17,Музыка и видео,https://www.ozon.ru/category/muzyka-i-video-13...,6
